In [1]:
!pip install -q "datasets<3" transformers seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import numpy as np
from collections import Counter
from datasets import load_dataset

dataset = load_dataset("nlpaueb/finer-139")
label_names = dataset["train"].features["ner_tags"].feature.names

counts = Counter()
for ex in dataset["train"]:
    counts.update(ex["ner_tags"])
total = sum(counts.values()); o_count = counts[0]
label_counts = sorted([(label_names[i], n) for i, n in counts.items() if i != 0],
                      key=lambda x: x[1], reverse=True)

print("=== Label distribution ===")
print(f"Total tokens: {total:,}")
print(f"'O' ratio: {o_count/total*100:.2f}%  |  Labeled ratio: {(total-o_count)/total*100:.2f}%")
print(f"Active labels in train: {len(label_counts)}")
print("Top 5:", label_counts[:5])

lengths = np.array([len(ex["tokens"]) for ex in dataset["train"]])
print(f"\nSentence length — mean {lengths.mean():.1f}, median {np.median(lengths):.0f}, "
      f"95th {np.percentile(lengths,95):.0f}, >128: {(lengths>128).mean()*100:.1f}%")

Generating train split:   0%|          | 0/900384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/112494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/108378 [00:00<?, ? examples/s]

=== Label distribution ===
Total tokens: 40,827,499
'O' ratio: 99.22%  |  Labeled ratio: 0.78%
Active labels in train: 167
Top 5: [('B-DebtInstrumentInterestRateStatedPercentage', 15208), ('B-LineOfCreditFacilityMaximumBorrowingCapacity', 11693), ('B-DebtInstrumentBasisSpreadOnVariableRate1', 11445), ('B-DebtInstrumentFaceAmount', 10749), ('B-AllocatedShareBasedCompensationExpense', 8710)]

Sentence length — mean 45.3, median 37, 95th 104, >128: 2.6%


In [3]:
import os
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def align_labels(example):
    tok = tokenizer(example["tokens"], truncation=True, max_length=128, is_split_into_words=True)
    word_ids = tok.word_ids(); labels = []; prev = None
    for wid in word_ids:
        if wid is None: labels.append(-100)
        elif wid != prev: labels.append(example["ner_tags"][wid])
        else: labels.append(-100)
        prev = wid
    tok["labels"] = labels
    return tok

tokenized = dataset.map(align_labels, remove_columns=dataset["train"].column_names,
                        desc="Tokenizing")

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
for s in ["train","validation","test"]:
    dataset[s].to_parquet(f"data/raw/{s}.parquet")
    tokenized[s].to_parquet(f"data/processed/{s}.parquet")
print("Parquet files saved.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/900384 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/113 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/113 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/109 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/109 [00:00<?, ?ba/s]

Parquet files saved.


In [4]:
readme = (
"# FiNER-139: Financial Numeric Entity Recognition (XBRL Tagging)\n\n"
"Token classification on SEC financial filings. ~95% of labeled tokens are numbers, "
"so the correct XBRL tag depends on context, not the number itself.\n\n"
"## Dataset\n"
"- Source: nlpaueb/finer-139 (ACL 2022)\n"
"- 1.1M sentences: 900,384 train / 112,494 val / 108,378 test\n"
"- 139 entity types -> 279 BIO tags; 167 active in train\n"
"- data/raw/ = original, data/processed/ = tokenized + aligned\n\n"
"## Key EDA facts\n"
"- 99.22% O vs 0.78% labeled (severe imbalance)\n"
"- 94.7% of labeled tokens numeric\n"
"- 10.6% of words split into sub-pieces (number fragmentation)\n"
"- max_length=128 covers 97.4% of sentences\n\n"
"## Evaluation\nEntity-level F1 (seqeval); accuracy misleading due to 99% O majority.\n\n"
"## Team\nSee CONTRIBUTORS.md.\n")
open("README.md","w").write(readme)
open("CONTRIBUTORS.md","w").write("# Contributors\n\n<!-- name | github | role -->\n")
open("requirements.txt","w").write("datasets<3\ntransformers\ntorch\nseqeval\nnumpy\npandas\nscikit-learn\n")
open(".gitignore","w").write("__pycache__/\n*.py[cod]\n.ipynb_checkpoints/\n.DS_Store\n*.log\n")
print("Project files created.")

Project files created.


In [ ]:
import zipfile, os
files = ["README.md","CONTRIBUTORS.md","requirements.txt",".gitignore",
         "data/raw/train.parquet","data/raw/validation.parquet","data/raw/test.parquet",
         "data/processed/train.parquet","data/processed/validation.parquet","data/processed/test.parquet"]
with zipfile.ZipFile("finer139_repo.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f): z.write(f)
print("ZIP ready:", round(os.path.getsize("finer139_repo.zip")/1024/1024,1), "MB")